In [ ]:
import numpy as np

class Board():
    def __init__(self):
        self.state = np.zeros((51,51))
        for j in range (51): 
            self.state[0,j] = -1
            self.state[50,j] = -1
        for i in range (51):
            self.state[i,0]= -1
            self.state[i,50]= -1
    def update(self, row, col):
        if (self.state[row, col] == 2):
            return True
        else:
            self.state[row, col] = 2
            return False
    def comida(self,row, col):
        if self.state[row, col] == 0:
            self.state[row, col] = 1
    def is_game_over(self):
        for j in range(51):
            f1_sum = sum(self.state[0,j])
            f2_sum = sum(self.state[50,j])
        for i in range(51):
            c1_sum = sum(self.state[i,0])
            c2_sum = sum(self.state[i,50])
        if(f1_sum > -51 or f2_sum> -51):
            return -1
        else: 
            if(c1_sum > -51 or c2_sum> -51):
                return -1
            else:
                if self.update():
                    return -1
                else:
                    return 1
    def reset(self):
        self.state = np.zeros((51,51))
        for j in range (51): 
            self.state[0,j] = -1
            self.state[50,j] = -1
        for i in range (51):
            self.state[i,0]= -1
            self.state[i,50]= -1


In [ ]:
from tqdm import tqdm

class Game():
    def __init__(self, player1):
        player1.symbol = 2
        self.player = player1
        self.board = Board()
        
    def selfplay(self, rounds=200):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            player = self.player
            player.reset()
            game_over = False
            while not game_over:
                action = player.move(self.board)
                self.board.update(player.symbol, action[0], action[1])
                    for p in self.player:
                        p.update(self.board)
                    ganador = self.board.is_game_over()
                    if ganador is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.player):
                if ganador == player.symbol:
                    wins[ix] += 1
        return wins


    def reward(self):
        winner = self.board.is_game_over()
        if winner == 0: # empate
            for player in self.player:
                player.reward(0.5) #dividimos la recommpensa
        else: # le damos 1 recompensa al jugador que gana y le quitamos sus ahorros al que pierde
            for player in self.player:
                if winner == player.symbol:
                    player.reward(1)
                else:
                    player.reward(-2)

In [ ]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {} # tabla con pares estado -> valor
        self.alpha = alpha         # learning rate
        self.positions = []       # guardamos todas las posiciones de la partida
        self.prob_exp = prob_exp   # probabilidad de explorar
        self.symbol = None

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        # exploracion
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            # vamos a una posición aleatoria
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]
        # explotacion
        # vamos a la posición con más valor
        max_value = -1000
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(-1))
            v = self.value_function.get(next_state)
            value = 0 if v is None else v
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col
        return best_row, best_col

    def update(self, board):
        self.positions.append(str(board.state.reshape(-1)))


    def reward(self, reward):
        # al final de la partida (cuando recibimos la recompensa)
        # iteramos por tods los estados actualizando su valor en la tabla
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]